# Jalon 2 — Apprentissage de 2 modèles MR-Sort (inv-2-MR-Sort) avec lincs

Chaque jeu de données mélange deux segments de clients qui ne se comportent pas de la même façon. On cherche simultanément une partition du learning set en 2 segments et un modèle MR-Sort par segment. Les modèles sont appris avec l'heuristique de la bibliothèque lincs.

**Hypothèses**
- Les features sont dans $[0, 1]$ et à maximiser (1 est meilleur que 0).
- Les catégories sont ordonnées de façon croissante : la première est la pire, la dernière la meilleure.
- La clientèle est composée de **2 segments**, et l'on ne sait pas à quel segment appartient chaque objet.
- Les jeux de données diffèrent par le nombre de critères (6, 8 ou 10), le nombre de catégories (3, 4 ou 5), la taille (300, 600 ou 1200), la répartition entre segments (50-50 ou 75-25) et le bruit (0 % ou 10 %).

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lincs.classification as lc

DATA_DIR = Path("Datasets-Step2")

## 1. Chargement des données

Chaque dossier contient un `problem.yml`, qui décrit les critères et les catégories, et des fichiers CSV qui forment le learning set. Les `problem.yml` fournis n'ont pas le champ `kind` exigé par lincs 2 : on l'ajoute au moment du chargement. La classe `y` d'un objet est l'indice de sa catégorie dans l'ordre croissant.

In [ ]:
def load_dataset(csv_path):
    csv_path = Path(csv_path)
    yml = "kind: classification-problem\n" + (csv_path.parent / "problem.yml").read_text()
    problem = lc.Problem.load(io.StringIO(yml))
    df = pd.read_csv(csv_path)
    categories = [c.name for c in problem.ordered_categories]
    y = df["category"].astype(str).map(categories.index).to_numpy()
    return problem, df, y

def to_alternatives(problem, df):
    return lc.Alternatives.load(problem, io.StringIO(df.to_csv(index=False)))


print(load_dataset(DATA_DIR / "type_a/cr06_ca03/n0300_rep50-50_noise00.csv"))

## 2. Apprentissage d'un modèle MR-Sort avec lincs

On utilise l'heuristique *weights-profiles-breed* de lincs (thèse d'O. Sobrie) avec ses réglages par défaut : une population de 9 modèles, et à chaque itération on optimise les poids par programmation linéaire puis on améliore les profils.

L'apprentissage s'arrête dès que l'une de ces conditions est remplie :
- le sous-ensemble est restitué à 100 % ;
- `max_seconds` secondes se sont écoulées ;
- 20 itérations se sont succédé sans progrès.

In [ ]:
def learn_mrsort(problem, df, seed=0, max_seconds=2):
    pls = lc.PreprocessedLearningSet(problem, to_alternatives(problem, df))
    models = lc.LearnMrsortByWeightsProfilesBreed.ModelsBeingLearned(pls, 9, seed)
    init = lc.InitializeProfilesForProbabilisticMaximalDiscriminationPowerPerCriterion(pls, models)
    weights = lc.OptimizeWeightsUsingGlop(pls, models)
    profiles = lc.ImproveProfilesWithAccuracyHeuristicOnCpu(pls, models)
    breed = lc.ReinitializeLeastAccurate(models, init, weights, 4)
    termination = lc.TerminateWhenAny([
        lc.TerminateAtAccuracy(models, len(df)),
        lc.TerminateAfterSeconds(max_seconds),
        lc.TerminateAfterIterationsWithoutProgress(models, 20),
    ])
    return lc.LearnMrsortByWeightsProfilesBreed(pls, models, init, weights, profiles, breed, termination).perform()

def predict(problem, model, df):
    alternatives = to_alternatives(problem, df)
    lc.classify_alternatives(problem, model, alternatives)
    return np.array([a.category_index for a in alternatives.alternatives])

### Marge d'un objet pour un modèle

Prenons un objet $x$ de catégorie $h$ et notons $\sigma(x, b)$ la somme des poids des critères sur lesquels $x$ atteint le profil $b$. La marge de $x$ pour un modèle est :
$$m(x) = \min\big(\sigma(x, b^{h-1}) - \lambda,\ \ \lambda - \sigma(x, b^{h})\big)$$
Elle est positive si et seulement si $x$ est bien classé, et d'autant plus grande que le classement est robuste. Elle est négative s'il est mal classé, et d'autant plus négative que l'erreur est grande.

lincs fixe le seuil de majorité à 1 avec des poids non normalisés. On divise donc la marge par la somme des poids pour pouvoir comparer les deux modèles. lincs calcule en float32 : on fait de même, avec une petite tolérance.

In [ ]:
def margins(problem, model, df, y, tol=1e-6):
    X = df[[c.name for c in problem.criteria]].to_numpy(dtype=np.float32)
    w = np.array(model.sufficient_coalitions[0].weights.criterion_weights, dtype=np.float64)
    b = np.array([[np.inf if t is None else t for t in av.real_thresholds.thresholds]
                  for av in model.accepted_values], dtype=np.float32).T     # (p-1, n)
    sigma = (X[:, None, :] >= b[None]) @ w                                   # (N, p-1)
    p, rows = b.shape[0] + 1, np.arange(len(y))
    low = np.where(y >= 1, sigma[rows, np.maximum(y - 1, 0)] - (1 - tol), np.inf)
    up = np.where(y <= p - 2, (1 - tol) - sigma[rows, np.minimum(y, p - 2)], np.inf)
    return np.minimum(low, up) / w.sum()

## 3. Heuristique pour 2 segments

1. **Initialisation** : chaque objet est affecté au hasard au segment 0 ou au segment 1.
2. **Apprentissage** : on apprend un modèle MR-Sort $\mathcal{M}_k$ sur chaque segment $S_k$.
3. **Évaluation** : un objet est *bien appris* si le modèle de son propre segment lui attribue sa vraie catégorie. On mesure la précision, c'est-à-dire la proportion d'objets bien appris.
4. **Réaffectation par la marge** : chaque objet, bien ou mal appris, change de segment si le modèle de l'autre segment le classe avec une marge **strictement plus grande**. En cas d'égalité, il reste où il est.
   - Un objet bien classé par les deux modèles rejoint celui qui le classe le plus solidement.
   - Un objet mal classé par les deux rejoint celui qui se trompe le moins.
   - Un objet ne bouge que si sa marge augmente, ce qui évite les allers-retours. C'est la règle de réaffectation de l'algorithme ACC (Brisimi et al., 2018).
5. On recommence à l'étape 2 jusqu'à ce que la précision atteigne le seuil, ou jusqu'à `max_iter` itérations.

La précision n'augmente pas forcément à chaque itération, car lincs est une heuristique aléatoire. On renvoie donc la meilleure itération rencontrée.

In [ ]:
def learn_2_mrsort(problem, df, y, threshold=0.95, max_iter=20, max_seconds=2, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(df))
    segment = rng.integers(0, 2, len(df))                      # 1. partition aléatoire
    history, best = [], None

    for it in range(max_iter):
        models = [learn_mrsort(problem, df[segment == k], seed + 2 * it + k, max_seconds)   # 2.
                  for k in (0, 1)]
        pred = np.array([predict(problem, m, df) for m in models])                         # (2, N)
        well_learned = pred[segment, idx] == y                                              # 3.
        accuracy = well_learned.mean()
        history.append(accuracy)
        if best is None or accuracy > best["accuracy"]:
            best = dict(accuracy=accuracy, models=models, segment=segment.copy(), iteration=it)
        if accuracy >= threshold:
            break
        margin = np.array([margins(problem, m, df, y) for m in models])                     # (2, N)
        move = margin[1 - segment, idx] > margin[segment, idx]                              # 4.
        segment = np.where(move, 1 - segment, segment)

    best["history"] = history
    return best

## 4. Exemple sur un jeu de données

On compare avec un seul modèle MR-Sort appris sur tout le learning set, c'est-à-dire sans segmentation.

In [ ]:
problem, df, y = load_dataset(DATA_DIR / "type_a/cr06_ca03/n0300_rep50-50_noise00.csv")

single = learn_mrsort(problem, df)
print(f"1 modèle MR-Sort  : précision = {np.mean(predict(problem, single, df) == y):.3f}")

result = learn_2_mrsort(problem, df, y)
print(f"2 modèles MR-Sort : précision = {result['accuracy']:.3f} "
      f"(itération {result['iteration'] + 1}, tailles des segments = {np.bincount(result['segment'], minlength=2)})")

plt.plot(range(1, len(result["history"]) + 1), result["history"], marker="o")
plt.axhline(0.95, ls="--", c="grey", label="seuil")
plt.xlabel("itération"); plt.ylabel("proportion d'objets bien appris"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
for k, model in enumerate(result["models"]):
    print(f"--- Modèle du segment {k}")
    print("\n".join(lc.describe_model(problem, model)))

## 5. Résultats sur tous les jeux de données

Avec 10 % de bruit, le seuil de 0.95 ne peut en général pas être atteint : c'est alors `max_iter` qui arrête la boucle.

In [ ]:
rows = []
for csv_path in sorted(DATA_DIR.glob("type_*/cr*_ca*/*.csv")):
    problem, df, y = load_dataset(csv_path)
    single_acc = np.mean(predict(problem, learn_mrsort(problem, df), df) == y)
    res = learn_2_mrsort(problem, df, y)
    rows.append(dict(type=csv_path.parent.parent.name, dossier=csv_path.parent.name, fichier=csv_path.stem,
                     precision_1_modele=round(single_acc, 3), precision_2_modeles=round(res["accuracy"], 3),
                     iterations=len(res["history"]), segments=tuple(np.bincount(res["segment"], minlength=2))))

results = pd.DataFrame(rows)
results

In [ ]:
results.groupby(["type", "dossier"])[["precision_1_modele", "precision_2_modeles"]].mean().round(3)